# `flare_04` — stage AF-divergent eval panels (Part 8 prep)

Builds the fixed marker panels that `flare_02` Part 8 needs for allele–ancestry
and Mendelian scoring, then uploads them to the workspace bucket.

| Panel | Region | Dest |
|---|---|---|
| `chr22_10mb` | `chr22:26897597-36897597` | `$WORKSPACE_BUCKET/refs/flare/eval_panels/` |
| `chr20_full` | full `chr20` | same |

**Products:** `{label}.markers.tsv` (required by Part 8), plus `{label}.bed.gz`
(+ `.tbi` when tabix works).

## How to run

1. Prefer a fresh `00_sync_repo` so `scripts/flare_build_af_panel.py` is current.
2. Run cells top → bottom. Chr20 takes longest (full-chromosome AF scan).
3. Re-run `flare_02` Part 8; it will pull panels from the bucket into `OUT/eval_panels/`.

Defaults match `flare/configs/lai_exp.tsv` gnomAD LAI refs. Override with env
vars or edit the config cell. Set `FLARE_PANEL_DRY_RUN=true` to skip the upload.


## Bootstrap

Find an **up-to-date** `scripts/flare_build_af_panel.py` (must contain the
`bcftools view | query` Terra fix). Prefers `aou-lr-phase-2/scripts` over
stale `$WORKSPACE/edit/scripts` copies from an older sync.


In [ ]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

# Prefer the git clone over $WORKSPACE/edit/scripts (staged copies go stale).
candidates = [
    Path(os.environ["AOU_LR_REPO_DIR"]) / "scripts" if os.environ.get("AOU_LR_REPO_DIR") else None,
    Path.cwd() / "aou-lr-phase-2" / "scripts",
    Path.cwd().parent / "aou-lr-phase-2" / "scripts",
    Path.home() / "AoU_DRC_LongReads_PhaseTwo_Storage" / "edit" / "aou-lr-phase-2" / "scripts",
    Path.cwd() / "scripts",
    Path.cwd().parent / "scripts",
]
candidates = [p for p in candidates if p is not None]

def _script_ok(path: Path) -> bool:
    """Require the Terra-safe view|query builder (not old query -m2)."""
    text = path.read_text(errors="replace")
    if "query: invalid option" in text:
        return False
    # Marker from c40b989+
    return "bcftools view" in text and "Terra-safe" in text


SCRIPTS = None
for _d in candidates:
    cand = _d / "flare_build_af_panel.py"
    if not cand.is_file():
        continue
    if not _script_ok(cand):
        print(f"skip stale builder: {cand}", flush=True)
        continue
    SCRIPTS = _d.resolve()
    break

if SCRIPTS is None:
    bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not bucket:
        raise FileNotFoundError(
            "No up-to-date flare_build_af_panel.py found. "
            "Run 00_sync_repo (or git pull in aou-lr-phase-2) so scripts include the view|query fix."
        )
    local = Path("/tmp/aou_lr_scripts")
    local.mkdir(parents=True, exist_ok=True)
    dest = local / "flare_build_af_panel.py"
    subprocess.check_call(
        ["gsutil", "-q", "cp", f"{bucket}/scripts/flare_build_af_panel.py", str(dest)]
    )
    if not _script_ok(dest):
        raise SystemExit(
            f"Bucket copy is also stale ({dest}). Re-run 00_sync_repo after pulling "
            "main (>= c40b989), then re-open this notebook."
        )
    SCRIPTS = local

builder = SCRIPTS / "flare_build_af_panel.py"
assert _script_ok(builder), builder

if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

print("SCRIPTS:", SCRIPTS)
print("builder:", builder)
print("bcftools:", shutil.which("bcftools") or "MISSING")
print("gsutil:", shutil.which("gsutil") or "MISSING")
assert shutil.which("bcftools"), "bcftools required"
assert shutil.which("gsutil"), "gsutil required"


## Config


In [ ]:
BUCKET = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
assert BUCKET.startswith("gs://"), "WORKSPACE_BUCKET must be set on Terra"

# Defaults = paths used by flare/configs/lai_exp.tsv
REF_PANEL = os.environ.get(
    "REF_PANEL",
    "gs://fc-secure-1c6b9393-5e5d-4a87-b483-d1f0b019af92/Reference_plink/aou_1000genomes.refmap",
)
REF_VCF_CHR22 = os.environ.get(
    "REF_VCF_CHR22",
    "gs://fc-secure-1c6b9393-5e5d-4a87-b483-d1f0b019af92/outputs/gnomad_lai_90/chr22.gnomad_lai_90.vcf.bgz",
)
REF_VCF_CHR20 = os.environ.get(
    "REF_VCF_CHR20",
    "gs://fc-secure-1c6b9393-5e5d-4a87-b483-d1f0b019af92/outputs/gnomad_lai_90/chr20.gnomad_lai_90.vcf.bgz",
)
CHR20_REGION = os.environ.get("CHR20_REGION", "chr20")
CHR22_REGION = os.environ.get("CHR22_REGION", "chr22:26897597-36897597")

TOP_N = int(os.environ.get("TOP_N", "5000"))
MIN_MAC = int(os.environ.get("MIN_MAC", "50"))
MIN_AF_RANGE = float(os.environ.get("MIN_AF_RANGE", "0.05"))

BUILD_CHR22 = os.environ.get("BUILD_CHR22", "true").lower() in {"1", "true", "yes", "on"}
BUILD_CHR20 = os.environ.get("BUILD_CHR20", "true").lower() in {"1", "true", "yes", "on"}
DRY_RUN = os.environ.get("FLARE_PANEL_DRY_RUN", "").lower() in {"1", "true", "yes", "on"}
FORCE_RELOCALIZE = os.environ.get("FORCE_RELOCALIZE", "").lower() in {"1", "true", "yes", "on"}

OUT_LOCAL = Path(os.environ.get("OUT_LOCAL", "/tmp/flare_eval_panels"))
CACHE_LOCAL = Path(os.environ.get("CACHE_LOCAL", "/tmp/flare_eval_refs"))
DEST = f"{BUCKET}/refs/flare/eval_panels"
OUT_LOCAL.mkdir(parents=True, exist_ok=True)
CACHE_LOCAL.mkdir(parents=True, exist_ok=True)

print("BUCKET:", BUCKET)
print("DEST:", DEST)
print("OUT_LOCAL:", OUT_LOCAL)
print("CACHE_LOCAL:", CACHE_LOCAL)
print("BUILD_CHR22:", BUILD_CHR22, CHR22_REGION)
print("BUILD_CHR20:", BUILD_CHR20, CHR20_REGION)
print("TOP_N / MIN_MAC / MIN_AF_RANGE:", TOP_N, MIN_MAC, MIN_AF_RANGE)
print("DRY_RUN:", DRY_RUN)


## Localize refmap + VCFs (+ `.tbi`)

Avoids pathlib collapsing `gs://` → `gs:/`. Skips files that already exist unless
`FORCE_RELOCALIZE`.


In [ ]:
def localize(uri: str, dest: Path) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if uri.startswith("gs://"):
        if FORCE_RELOCALIZE or not dest.is_file() or dest.stat().st_size == 0:
            print(f"gsutil cp {uri} -> {dest}", flush=True)
            subprocess.check_call(["gsutil", "-q", "cp", uri, str(dest)])
        else:
            print(f"cached {dest}", flush=True)
        if uri.endswith((".vcf.gz", ".vcf.bgz")):
            tbi = Path(str(dest) + ".tbi")
            if FORCE_RELOCALIZE or not tbi.is_file() or tbi.stat().st_size == 0:
                try:
                    subprocess.check_call(["gsutil", "-q", "cp", uri + ".tbi", str(tbi)])
                    print(f"cached index {tbi}", flush=True)
                except subprocess.CalledProcessError:
                    print(f"warning: no .tbi at {uri}.tbi", flush=True)
        return dest
    p = Path(uri)
    assert p.is_file(), p
    return p


REF_PANEL_LOCAL = localize(REF_PANEL, CACHE_LOCAL / Path(REF_PANEL).name)
REF_VCF_CHR22_LOCAL = localize(REF_VCF_CHR22, CACHE_LOCAL / Path(REF_VCF_CHR22).name) if BUILD_CHR22 else None
REF_VCF_CHR20_LOCAL = localize(REF_VCF_CHR20, CACHE_LOCAL / Path(REF_VCF_CHR20).name) if BUILD_CHR20 else None
print("REF_PANEL_LOCAL:", REF_PANEL_LOCAL)
print("REF_VCF_CHR22_LOCAL:", REF_VCF_CHR22_LOCAL)
print("REF_VCF_CHR20_LOCAL:", REF_VCF_CHR20_LOCAL)


## Build panels

Calls `flare_build_af_panel.py` (view|query path; BED sorted for tabix).
Logs stay in the cell; chr20 may run 30–90+ minutes depending on VM.


In [ ]:
def build_panel(label: str, region: str, ref_vcf: Path) -> Path:
    prefix = OUT_LOCAL / label
    markers = Path(str(prefix) + ".markers.tsv")
    cmd = [
        sys.executable,
        str(SCRIPTS / "flare_build_af_panel.py"),
        "--ref-vcf", str(ref_vcf),
        "--ref-panel", str(REF_PANEL_LOCAL),
        "--region", region,
        "--top-n", str(TOP_N),
        "--min-mac", str(MIN_MAC),
        "--min-af-range", str(MIN_AF_RANGE),
        "--out-prefix", str(prefix),
    ]
    print("===", label, region, flush=True)
    print("+", " ".join(cmd), flush=True)
    subprocess.check_call(cmd)
    assert markers.is_file() and markers.stat().st_size > 0, markers
    n = sum(1 for _ in markers.open()) - 1
    print(f"ok {markers} ({n} markers)", flush=True)
    return markers


built = {}
if BUILD_CHR22:
    built["chr22_10mb"] = build_panel("chr22_10mb", CHR22_REGION, REF_VCF_CHR22_LOCAL)
if BUILD_CHR20:
    built["chr20_full"] = build_panel("chr20_full", CHR20_REGION, REF_VCF_CHR20_LOCAL)

print("built:", {k: str(v) for k, v in built.items()})
display({k: v.stat().st_size for k, v in built.items()})


## Upload to bucket

`gsutil -m rsync` → `$WORKSPACE_BUCKET/refs/flare/eval_panels/`.


In [ ]:
if DRY_RUN:
    print("DRY_RUN: skip upload; local panels in", OUT_LOCAL)
else:
    print(f"rsync {OUT_LOCAL}/ -> {DEST}/", flush=True)
    subprocess.check_call(["gsutil", "-m", "rsync", "-r", f"{OUT_LOCAL}/", f"{DEST}/"])
    print("listed remote:", flush=True)
    subprocess.check_call(["gsutil", "ls", "-lh", f"{DEST}/"])

print("Next: re-run flare_02 Part 8 (panels pull automatically from DEST).")
